# Data Preprocessing Pipeline

Loads the GUIDE-seq and CHANGE-seq datasets, cleans and validates sequence strings, collapses repeated GUIDE-seq measurements, splits train/test sets by gRNA to avoid data leakage, and saves the encoded features to disk.

In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import GroupShuffleSplit

# Paths
GUIDE_POSITIVE_PATH = 'SysEvalOffTarget/files/datasets/include_on_targets/GUIDEseq_positive.csv'
GUIDE_NEGATIVE_PATH = 'SysEvalOffTarget/files/datasets/include_on_targets/GUIDEseq_negative.csv'
CHANGE_POSITIVE_PATH = 'SysEvalOffTarget/files/datasets/include_on_targets/CHANGEseq_positive.csv'
CHANGE_NEGATIVE_PATH = 'SysEvalOffTarget/files/datasets/include_on_targets/CHANGEseq_negative.csv'

## Loading and Inspecting Raw Datasets

In [2]:
# Loading positive and negative sets generated by prepare_data.py and combining them into full datasets
guide_seq = pd.concat([pd.read_csv(GUIDE_POSITIVE_PATH), pd.read_csv(GUIDE_NEGATIVE_PATH)], ignore_index=True)
change_seq = pd.concat([pd.read_csv(CHANGE_POSITIVE_PATH), pd.read_csv(CHANGE_NEGATIVE_PATH)], ignore_index=True)

# Structure
for name, df in [('GUIDE-seq', guide_seq), ('CHANGE-seq', change_seq)]:
    print(f'--- {name} ---')
    print(f'Shape: {df.shape}')
    print(f'Columns: {list(df.columns)}')
    print(f'Dtypes:\n{df.dtypes}')
    print(f'Missing values:\n{df.isnull().sum()}')
    print()

--- GUIDE-seq ---
Shape: (1478003, 13)
Columns: ['Unnamed: 0', 'chrom', 'chromStart', 'chromEnd', 'name', 'GUIDEseq_reads', 'strand', 'offtarget_sequence', 'genomic_coordinate', 'distance', 'target', 'run', 'label']
Dtypes:
Unnamed: 0              int64
chrom                  object
chromStart              int64
chromEnd              float64
name                   object
GUIDEseq_reads        float64
strand                 object
offtarget_sequence     object
genomic_coordinate     object
distance                int64
target                 object
run                   float64
label                   int64
dtype: object
Missing values:
Unnamed: 0                  0
chrom                       0
chromStart                  0
chromEnd              1476301
name                  1476301
GUIDEseq_reads        1476301
strand                      0
offtarget_sequence          0
genomic_coordinate    1476301
distance                    0
target                      0
run                   1476

In [3]:
# Dropping uninformative columns
guide_seq = guide_seq.drop(columns=['Unnamed: 0'])
change_seq = change_seq.drop(columns=['Unnamed: 0', 'Unnamed: 7', 'chromStart:chromEnd'])

In [4]:
# Quick visual inspection of guide_seq data
guide_seq.head()

,chrom,chromStart,chromEnd,name,GUIDEseq_reads,strand,offtarget_sequence,genomic_coordinate,distance,target,run,label
0,chr19,55115744,55115767.0,AAVS1_site_1,13557.0,+,GTCACCAATCCTGTCCCTAGTGG,chr19:55115745-55115767:+,0,GTCACCAATCCTGTCCCTAGNGG,1.0,1
1,chrX,1450701,1450724.0,AAVS1_site_1,190.0,+,CTCCCCAACCCCATCCCTAGGGG,chrX:1450702-1450724:+,5,GTCACCAATCCTGTCCCTAGNGG,1.0,1
2,chrX,1452125,1452148.0,AAVS1_site_1,105.0,+,CTCCCCAACCCCATCCCTAGGGG,chrX:1452126-1452148:+,5,GTCACCAATCCTGTCCCTAGNGG,1.0,1
3,chr1,12523844,12523867.0,AAVS1_site_1,2.0,+,CACACTAATCCTGTCCCCAGAGG,chr1:12523845-12523867:+,4,GTCACCAATCCTGTCCCTAGNGG,1.0,1
4,chr19,55115744,55115767.0,AAVS1_site_1,55079.0,+,GTCACCAATCCTGTCCCTAGTGG,chr19:55115745-55115767:+,0,GTCACCAATCCTGTCCCTAGNGG,2.0,1


In [5]:
# Quick visual inspection of change_seq data
change_seq.head()

,chrom,chromStart,chromEnd,name,CHANGEseq_reads,strand,offtarget_sequence,distance,target,label
0,chr4,121343302,121343325.0,AAVS1_site_1,540.0,+,ATCACCTATCCTATCCCTAAGGG,4,GTCACCAATCCTGTCCCTAGNGG,1
1,chr1,12523844,12523867.0,AAVS1_site_1,314.0,+,CACACTAATCCTGTCCCCAGAGG,4,GTCACCAATCCTGTCCCTAGNGG,1
2,chr8,66370990,66371013.0,AAVS1_site_1,258.0,-,AGCATAAATCCTGTCCCTAGGAG,5,GTCACCAATCCTGTCCCTAGNGG,1
3,chr9,134937838,134937861.0,AAVS1_site_1,226.0,-,AAAACCAAACCTGTCCCTAAAGG,5,GTCACCAATCCTGTCCCTAGNGG,1
4,chr15,36995651,36995674.0,AAVS1_site_1,130.0,+,TGATCCTATCCTGTCCCTAGAGG,5,GTCACCAATCCTGTCCCTAGNGG,1


In [6]:
GRNA_COL = 'target'
TARGET_COL = 'offtarget_sequence'

# Checking sequence lengths before filtering (both should be 23nt)
print('GUIDE-seq target lengths:')
print(guide_seq[GRNA_COL].str.len().value_counts())
print('\nGUIDE-seq offtarget lengths:')
print(guide_seq[TARGET_COL].str.len().value_counts())

print('\nCHANGE-seq target lengths:')
print(change_seq[GRNA_COL].str.len().value_counts())
print('\nCHANGE-seq offtarget lengths:')
print(change_seq[TARGET_COL].str.len().value_counts())

GUIDE-seq target lengths:
target
23    1478003
Name: count, dtype: int64

GUIDE-seq offtarget lengths:
offtarget_sequence
23    1478003
Name: count, dtype: int64

CHANGE-seq target lengths:
target
23    2873627
Name: count, dtype: int64

CHANGE-seq offtarget lengths:
offtarget_sequence
23    2873627
Name: count, dtype: int64


## Parsing and Aligning Sequences

Both the guide target and candidate off-target sequences are expected to contain 23 nucleotides (20nt protospacer + 3nt PAM).

In [7]:
# Standardise sequences: drop missing, uppercase, strip whitespace, filter to 23nt
def parse_sequences(df, grna_col, target_col):
    df = df.copy()

    # Drop missing rows
    df = df.dropna(subset=[grna_col, target_col])

    df[grna_col] = df[grna_col].str.upper().str.strip()
    df[target_col] = df[target_col].str.upper().str.strip()

    # Both sequences are 23nt (20nt + 3nt PAM)
    valid = (df[grna_col].str.len() == 23) & (df[target_col].str.len() == 23)
    n_dropped = (~valid).sum()
    if n_dropped > 0:
        print(f'Dropping {n_dropped} rows with unexpected sequence lengths')
    return df[valid].reset_index(drop=True)

guide_seq = parse_sequences(guide_seq, GRNA_COL, TARGET_COL)
change_seq = parse_sequences(change_seq, GRNA_COL, TARGET_COL)

print(f'GUIDE-seq after parsing: {guide_seq.shape}')
print(f'CHANGE-seq after parsing: {change_seq.shape}')

GUIDE-seq after parsing: (1478003, 12)
CHANGE-seq after parsing: (2873627, 10)


## Active and Inactive Off-Target Sites

Labels were assigned by the OrensteinLab data preparation pipeline. The datasets are highly imbalanced, with substantially more candidate inactive sites than experimentally identified active off-target sites.

- label 1 indicates an active off-target site. For CHANGE-seq, sites with read counts >100 were considered active; for GUIDE-seq, all experimentally identified off-target sites were considered active.
- label 0 indicates a candidate inactive off-target site generated using Cas-OFFinder after experimentally identified sites were removed.

Some active GUIDE-seq sites were detected in multiple experimental runs and therefore appear more than once in the source data. Because the models predict unique off-target sites rather than individual experimental runs, repeated measurements are collapsed to one row per genomic site before splitting. For repeated sites, the observation with the highest GUIDE-seq read count is retained.

In [8]:
SITE_KEY = [GRNA_COL, 'chrom', 'chromStart', 'strand']

# Check repeated GUIDE-seq measurements before collapsing them
duplicate_rows = guide_seq[
    guide_seq.duplicated(subset=SITE_KEY, keep=False)
]

assert (duplicate_rows['label'] == 1).all()
assert (
    duplicate_rows.groupby(SITE_KEY)[TARGET_COL]
    .nunique()
    .le(1)
    .all()
)

# CHANGE-seq should not contain repeated sites
assert change_seq.duplicated(subset=SITE_KEY).sum() == 0

n_removed = guide_seq.duplicated(subset=SITE_KEY).sum()

# Keep the highest-read observation for each repeated GUIDE-seq site
guide_seq = (
    guide_seq
    .sort_values(
        'GUIDEseq_reads',
        ascending=False,
        na_position='last',
        kind='stable'
    )
    .drop_duplicates(subset=SITE_KEY, keep='first')
    .sort_index()
    .reset_index(drop=True)
)

assert guide_seq.duplicated(subset=SITE_KEY).sum() == 0

print(f'Repeated GUIDE-seq rows removed: {n_removed:,}')
print(f'GUIDE-seq after deduplication: {guide_seq.shape}')
print(f'GUIDE-seq positives: {guide_seq["label"].sum():,}')

Repeated GUIDE-seq rows removed: 241
GUIDE-seq after deduplication: (1477762, 12)
GUIDE-seq positives: 1,461


In [9]:
# Confirming class distribution - severe imbalance expected given biological rarity of cleavage events
for name, df in [('GUIDE-seq', guide_seq), ('CHANGE-seq', change_seq)]:
    counts = df['label'].value_counts()
    pct = df['label'].value_counts(normalize=True) * 100
    print(f'{name} label distribution:')
    print(pd.DataFrame({'count': counts, 'percent': pct.round(1)}))
    print()

GUIDE-seq label distribution:
         count  percent
label                  
0      1476301     99.9
1         1461      0.1

CHANGE-seq label distribution:
         count  percent
label                  
0      2806151     97.7
1        67476      2.3



In [10]:
# Validate binary labels
assert set(guide_seq['label'].unique()).issubset({0, 1})
assert set(change_seq['label'].unique()).issubset({0, 1})

print('PASS - labels are binary')

PASS - labels are binary


In [11]:
# Checks the characters present in each sequence column
for name, df in [('GUIDE-seq', guide_seq), ('CHANGE-seq', change_seq)]:
    guide_chars = set(''.join(df[GRNA_COL].unique()))
    target_chars = set(''.join(df[TARGET_COL].unique()))

    print(f'{name} guide characters: {sorted(guide_chars)}')
    print(f'{name} off-target characters: {sorted(target_chars)}')

GUIDE-seq guide characters: ['A', 'C', 'G', 'N', 'T']
GUIDE-seq off-target characters: ['A', 'C', 'G', 'N', 'T']
CHANGE-seq guide characters: ['A', 'C', 'G', 'N', 'T']
CHANGE-seq off-target characters: ['A', 'C', 'G', 'N', 'T']


## gRNA-Based Train/Test Split

In [12]:
# gRNA-disjoint split to prevent data leakage
def grna_split(df, grna_col, test_size=0.2, random_state=2000):
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=test_size,
        random_state=random_state
    )
    groups = df[grna_col]
    train_idx, test_idx = next(splitter.split(df, groups=groups))

    train = df.iloc[train_idx].reset_index(drop=True)
    test = df.iloc[test_idx].reset_index(drop=True)
    return train, test


guide_train, guide_test = grna_split(guide_seq, GRNA_COL)
change_train, change_test = grna_split(change_seq, GRNA_COL)

# Checks that splitting has not lost or duplicated rows
assert len(guide_train) + len(guide_test) == len(guide_seq)
assert len(change_train) + len(change_test) == len(change_seq)

print('PASS - train/test row counts reconstruct full datasets')

for name, train, test in [
    ('GUIDE-seq', guide_train, guide_test),
    ('CHANGE-seq', change_train, change_test)
]:
    print(f'{name}: train={len(train)}, test={len(test)}')
    print(f'  Train gRNAs: {train[GRNA_COL].nunique()}, Test gRNAs: {test[GRNA_COL].nunique()}')

PASS - train/test row counts reconstruct full datasets
GUIDE-seq: train=1229148, test=248614
  Train gRNAs: 46, Test gRNAs: 12
CHANGE-seq: train=2352636, test=520991
  Train gRNAs: 88, Test gRNAs: 22


In [13]:
# Confirming zero gRNA overlap between train and test sets
def validate_split(train_df, test_df, grna_col, dataset_name):
    train_guides = set(train_df[grna_col])
    test_guides = set(test_df[grna_col])

    overlap = train_guides & test_guides

    assert len(overlap) == 0, (
        f'{dataset_name}: {len(overlap)} gRNAs appear in both train and test'
    )
    print(f'{dataset_name}: PASS - no gRNA overlap between train and test')

validate_split(
    guide_train,
    guide_test,
    GRNA_COL,
    'GUIDE-seq'
)

validate_split(
    change_train,
    change_test,
    GRNA_COL,
    'CHANGE-seq'
)

GUIDE-seq: PASS - no gRNA overlap between train and test
CHANGE-seq: PASS - no gRNA overlap between train and test


In [14]:
# Reports cross-dataset guide familiarity
guide_train_guides = set(guide_train[GRNA_COL])
change_test_guides = set(change_test[GRNA_COL])

shared_train_test_guides = guide_train_guides & change_test_guides

print(
    f'GUIDE train ∩ CHANGE test: '
    f'{len(shared_train_test_guides)} shared gRNAs'
)

GUIDE train ∩ CHANGE test: 12 shared gRNAs


## One-Hot Encoding

In [15]:
# N represents an unspecified nucleotide and is encoded as all zeros
NUCLEOTIDES = 'ACGT'

# One-hot encodes the guide and off-target sequences
def encode_onehot(df, grna_col, target_col):
    n_rows = len(df)
    grna_len = len(df[grna_col].iloc[0])
    target_len = len(df[target_col].iloc[0])

    def encode_column(sequences, seq_len):
        encoded = np.zeros((n_rows, seq_len, 4), dtype=np.float32)

        for i, nuc in enumerate(NUCLEOTIDES):
            for pos in range(seq_len):
                encoded[:, pos, i] = sequences.str[pos] == nuc

        return encoded.reshape(n_rows, -1)

    grna_encoded = encode_column(df[grna_col], grna_len)
    target_encoded = encode_column(df[target_col], target_len)

    return np.concatenate([grna_encoded, target_encoded], axis=1)


print('Encoding GUIDE-seq...')
X_guide_train = encode_onehot(guide_train, GRNA_COL, TARGET_COL)
X_guide_test = encode_onehot(guide_test, GRNA_COL, TARGET_COL)

print('Encoding CHANGE-seq...')
X_change_train = encode_onehot(change_train, GRNA_COL, TARGET_COL)
X_change_test = encode_onehot(change_test, GRNA_COL, TARGET_COL)

print(f'GUIDE-seq train shape: {X_guide_train.shape}')
print(f'GUIDE-seq test shape: {X_guide_test.shape}')
print(f'CHANGE-seq train shape: {X_change_train.shape}')
print(f'CHANGE-seq test shape: {X_change_test.shape}')

Encoding GUIDE-seq...
Encoding CHANGE-seq...
GUIDE-seq train shape: (1229148, 184)
GUIDE-seq test shape: (248614, 184)
CHANGE-seq train shape: (2352636, 184)
CHANGE-seq test shape: (520991, 184)


In [16]:
# Checks expected feature dimensions
assert X_guide_train.shape[1] == 184
assert X_guide_test.shape[1] == 184
assert X_change_train.shape[1] == 184
assert X_change_test.shape[1] == 184

print('PASS - one-hot feature dimensions are correct')

PASS - one-hot feature dimensions are correct


In [17]:
OUTPUT_DIR = 'data/processed/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save one-hot arrays for baseline models
np.save(OUTPUT_DIR + 'X_guide_train.npy', X_guide_train)
np.save(OUTPUT_DIR + 'X_guide_test.npy', X_guide_test)
np.save(OUTPUT_DIR + 'X_change_train.npy', X_change_train)
np.save(OUTPUT_DIR + 'X_change_test.npy', X_change_test)

# Save labels
np.save(OUTPUT_DIR + 'y_guide_train.npy', guide_train['label'].values)
np.save(OUTPUT_DIR + 'y_guide_test.npy', guide_test['label'].values)
np.save(OUTPUT_DIR + 'y_change_train.npy', change_train['label'].values)
np.save(OUTPUT_DIR + 'y_change_test.npy', change_test['label'].values)

# Save raw sequence dataframes for transformer models later
guide_train.to_csv(OUTPUT_DIR + 'guide_train.csv', index=False)
guide_test.to_csv(OUTPUT_DIR + 'guide_test.csv', index=False)
change_train.to_csv(OUTPUT_DIR + 'change_train.csv', index=False)
change_test.to_csv(OUTPUT_DIR + 'change_test.csv', index=False)

print('All files saved to', OUTPUT_DIR)

All files saved to data/processed/
